# Task 9 — Python Exploratory Data Analysis & Financial Analysis

**Project:** NovaMart Financial Analytics & Revenue Forecasting  
**Phase:** Task 9 — Python EDA  
**Data Source:** `data/processed/fact_sales_processed.csv`  

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Setup paths and aesthetics
BASE_DIR = Path("..")
csv_path = BASE_DIR / "data" / "processed" / "fact_sales_processed.csv"
fig_dir = BASE_DIR / "reports" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)

# Efficient Data Loading
print(f"Loading processed dataset from {csv_path}...")
df = pd.read_csv(csv_path, low_memory=False)
print(f"Loaded {len(df):,} transaction line items.")


## 1. Authoritative Baseline Reconciliation

Verifies 100.000% alignment against PostgreSQL Data Warehouse baseline.


In [ ]:
# Execute Authoritative Reconciliation Assertions
rows = len(df)
bk_cnt = df.duplicated(subset=['transaction_id', 'sku', 'event_type']).sum()
gross = df['gross_sales'].sum()
disc = df['discount_amount'].sum()
net = df['net_sales'].sum()
cogs = df['cogs'].sum()
gp = df['gross_profit'].sum()
gm_pct = (gp / net) * 100
ref_rows = (df['event_type'] == 'Refund').sum()
ref_amt = df[df['event_type'] == 'Refund']['net_sales'].abs().sum()

print("=== RECONCILIATION AUDIT ===")
print(f"Rows:                 {rows:,} (Expected: 970,838)")
print(f"Business Key Dups:    {bk_cnt} (Expected: 0)")
print(f"Net Revenue:          ${net:,.2f} (Expected: $46,595,173.27)")
print(f"COGS:                 ${cogs:,.2f} (Expected: $20,811,116.93)")
print(f"Gross Profit:         ${gp:,.2f} (Expected: $25,784,056.34)")
print(f"Gross Margin %:       {gm_pct:.4f}% (Expected: 55.3363%)")
print(f"Refund Rows:          {ref_rows:,} (Expected: 9,449)")

assert rows == 970838
assert bk_cnt == 0
assert round(net, 2) == 46595173.27
assert round(cogs, 2) == 20811116.93
assert round(gp, 2) == 25784056.34
assert round(gm_pct, 4) == 55.3363
assert ref_rows == 9449
print("\nSUCCESS: 100.000% RECONCILED TO AUTHORITATIVE BASELINE!")


## 2. Monthly Financial Time-Series & Growth Analysis


In [ ]:
df['year_month'] = df['transaction_date'].str.slice(0, 7)
m_summary = df.groupby('year_month').agg(
    gross_sales=('gross_sales', 'sum'),
    discounts=('discount_amount', 'sum'),
    net_sales=('net_sales', 'sum'),
    cogs=('cogs', 'sum'),
    gross_profit=('gross_profit', 'sum'),
    refund_lines=('event_type', lambda x: (x == 'Refund').sum())
).reset_index()

m_summary['gross_margin_pct'] = (m_summary['gross_profit'] / m_summary['net_sales']) * 100
m_summary['mom_net_growth'] = m_summary['net_sales'].pct_change() * 100

print(m_summary.to_string(index=False))


## 3. Product Category Profitability Analysis


In [ ]:
cat_summary = df.groupby('category').agg(
    line_items=('transaction_id', 'count'),
    units=('qty', 'sum'),
    gross_sales=('gross_sales', 'sum'),
    discounts=('discount_amount', 'sum'),
    net_sales=('net_sales', 'sum'),
    cogs=('cogs', 'sum'),
    gross_profit=('gross_profit', 'sum'),
    refund_lines=('event_type', lambda x: (x == 'Refund').sum())
).reset_index()

cat_summary['gross_margin_pct'] = (cat_summary['gross_profit'] / cat_summary['net_sales']) * 100
cat_summary['rev_share_pct'] = (cat_summary['net_sales'] / net) * 100
cat_summary = cat_summary.sort_values(by='net_sales', ascending=False)

print(cat_summary.to_string(index=False))


## 4. Customer Segment Analysis (Identified vs Anonymous Guests)


In [ ]:
cust_summary = df.groupby('customer_type').agg(
    distinct_customers=('customer_id', 'nunique'),
    orders=('transaction_id', 'nunique'),
    line_items=('transaction_id', 'count'),
    net_sales=('net_sales', 'sum'),
    gross_profit=('gross_profit', 'sum')
).reset_index()

cust_summary['gross_margin_pct'] = (cust_summary['gross_profit'] / cust_summary['net_sales']) * 100
cust_summary['aov'] = cust_summary['net_sales'] / cust_summary['orders']

print(cust_summary.to_string(index=False))


## 5. Top 10 Products by Net Revenue


In [ ]:
prod_top = df.groupby(['sku', 'item', 'category']).agg(
    units=('qty', 'sum'),
    net_sales=('net_sales', 'sum'),
    gross_profit=('gross_profit', 'sum')
).reset_index().sort_values(by='net_sales', ascending=False).head(10)

prod_top['gross_margin_pct'] = (prod_top['gross_profit'] / prod_top['net_sales']) * 100
print(prod_top.to_string(index=False))
